# 02. Preprocessing Views — 분석용 뷰 계층

`v_flight_analysis`는 `raw_departure_flight`의 모든 원문 컬럼을 보존하고 시간 파싱·기본 플래그·비파괴 문자열 표준화를 추가한다. 시간차는 보정 없는 원시값이며, 지연 플래그·자정 넘김 보정·음수값 클립은 만들지 않는다.

집계 통계는 집계 행만 제외한 MySQL 뷰로 유지한다. 실체화 테이블은 만들지 않는다.

## 0. 연결 설정

In [1]:
from pathlib import Path
import os
import re

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

load_dotenv(ROOT / '.env')
SQL_FILE = ROOT / 'sql' / '02_preprocessing_views.sql'

db_config = {
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT', '3306')),
    'database': os.getenv('DB_NAME'),
    'username': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
}
missing = [key for key in ['host', 'database', 'username', 'password'] if not db_config[key]]
if missing:
    raise ValueError(f'.env의 DB 설정이 비어 있음: {missing}')

engine = create_engine(URL.create('mysql+pymysql', **db_config), pool_pre_ping=True, future=True)

def load_named_queries(path: Path) -> dict[str, str]:
    sql_text = path.read_text(encoding='utf-8')
    matches = list(re.finditer(r'^-- name: ([a-z0-9_]+)\s*$', sql_text, flags=re.MULTILINE))
    return {
        match.group(1): sql_text[match.end(): matches[index + 1].start() if index + 1 < len(matches) else len(sql_text)].strip().rstrip(';')
        for index, match in enumerate(matches)
    }

SQL = load_named_queries(SQL_FILE)

def query(name: str) -> pd.DataFrame:
    return pd.read_sql(text(SQL[name]), engine)

def execute_ddl(name: str) -> None:
    with engine.begin() as connection:
        connection.execute(text(SQL[name]))

with engine.connect() as connection:
    connection.execute(text('SELECT 1'))

print(f"Connected to MySQL database: {db_config['database']}")

Connected to MySQL database: korea_outbound_travel


## 1. 분석용 뷰 생성

In [2]:
# 실행 순서를 고정해 참조 뷰, 분석 뷰, 기존 호환 뷰, 월간 집계 뷰를 생성·갱신한다.
DDL_NAMES = [
    'create_v_domestic_airport_reference',
    'create_v_destination_alias_reference',
    'create_v_route_monthly',
    'create_v_international_destination_reference',
    'create_v_flight_analysis',
    'create_v_departure_flight_compatibility',
    'create_v_airline_monthly',
    'create_v_airport_monthly',
    'create_v_country_monthly',
]
for ddl_name in DDL_NAMES:
    execute_ddl(ddl_name)

# named query 이름과 별개로 실제 생성되는 MySQL 뷰 이름을 표시한다.
VIEW_NAMES = [
    'v_domestic_airport_reference',
    'v_destination_alias_reference',
    'v_international_destination_reference',
    'v_flight_analysis',
    'v_departure_flight',
    'v_route_monthly',
    'v_airline_monthly',
    'v_airport_monthly',
    'v_country_monthly',
]
display(pd.DataFrame({'생성·갱신한 뷰': VIEW_NAMES}))

,생성·갱신한 뷰
0,v_domestic_airport_reference
1,v_destination_alias_reference
2,v_international_destination_reference
3,v_flight_analysis
4,v_departure_flight
5,v_route_monthly
6,v_airline_monthly
7,v_airport_monthly
8,v_country_monthly


## 2. 행 보존·날짜·시간 파싱 검증

In [3]:
# 중복 후보는 원문 컬럼 차이가 있어 삭제하지 않는다.
display(query('flight_duplicate_integrity_summary'))

,duplicate_group_count,duplicate_row_count,identical_group_count,differing_group_count
0,150,300.0,0.0,150.0


In [4]:
# 모든 분석용 뷰가 집계 행 제외 후 원본 행을 보존하는지 확인한다.
row_check = query('view_row_count_check')
display(row_check)
diff_columns = ['primary_row_difference', 'secondary_row_difference']
unmatched_views = row_check[row_check[diff_columns].fillna(0).ne(0).any(axis=1)]
if not unmatched_views.empty:
    raise AssertionError(f'행 수가 보존되지 않은 뷰: {unmatched_views["view_name"].tolist()}')

,view_name,raw_primary_rows,raw_secondary_rows,excluded_primary_rows,excluded_secondary_rows,view_rows,primary_row_difference,secondary_row_difference
0,v_flight_analysis,793865,NaN,0,NaN,793865,0,NaN
1,v_route_monthly,9670,9670.0,36,36.0,9634,0,0.0
2,v_airline_monthly,3951,3951.0,108,108.0,3843,0,0.0
3,v_airport_monthly,301,NaN,36,NaN,265,0,NaN
4,v_country_monthly,2083,NaN,36,NaN,2047,0,NaN


In [5]:
# 시간 파싱은 v_flight_analysis 생성 뒤 이 단계에서만 집계·검증한다.
display(query('time_calculable_summary'))

,airport_name_raw,view_rows,time_calculable_rows,time_calculable_rate
0,김포,199127,196369.0,0.9861
1,인천,594738,584322.0,0.9825


## 3. 파생 플래그·문자열 표준화 검증

In [6]:
# 플래그 건수·비율과 상태 미상 행을 확인한다. is_denominator는 상태값 기반 후보 플래그다.
flag_summary = query('flight_analysis_flag_summary')
display(flag_summary)

# 현재 확인된 상태값 플래그가 전체 행을 빠짐없이 덮는지 검사한다.
row = flag_summary.iloc[0]
covered_status_rows = (row['denominator_rows'] + row['cancelled_rows']
                       + row['diverted_rows'] + row['status_unknown_rows'])
if covered_status_rows != row['total_rows']:
    raise AssertionError(f'상태 플래그 미포함 행 존재: {covered_status_rows} vs {row["total_rows"]}')

,total_rows,passenger_rows,passenger_rate,denominator_rows,denominator_rate,cancelled_rows,cancelled_rate,diverted_rows,diverted_rate,status_unknown_rows,status_unknown_rate,time_calculable_rows,time_calculable_rate,domestic_reference_match_rows,domestic_reference_nonmatch_rows,domestic_reference_unknown_rows,domestic_rows,international_rows,unresolved_rows,reference_conflict_rows
0,793865,710755.0,0.8953,783204.0,0.9866,6586.0,0.0083,208.0,0.0003,3867.0,0.0049,780691.0,0.9834,174354.0,619497.0,14.0,174354.0,601999.0,17512.0,0.0


In [7]:
# 전체 로그와 여객편의 국내·국제·미확정·충돌 분류 범위를 확인한다.
display(query('flight_scope_classification_summary'))

,flight_scope_std,rows_count,row_share,passenger_rows,passenger_share,international_flag_true_rows,international_flag_false_rows,international_flag_null_rows
0,international,601999,0.7583,535442.0,0.7533,601999.0,0.0,0.0
1,domestic,174354,0.2196,174130.0,0.2450,0.0,174354.0,0.0
2,unresolved,17512,0.0221,1183.0,0.0017,0.0,0.0,17512.0


In [8]:
# 공항·서비스 구분별 분류 결과를 확인한다.
display(query('flight_scope_airport_service_profile'))

,airport_name_raw,service_type_raw,flight_scope_std,rows_count,share_within_airport_service
0,김포,기타,international,22,0.1606
1,김포,기타,domestic,94,0.6861
2,김포,기타,unresolved,21,0.1533
3,김포,여객,international,30899,0.1553
4,김포,여객,domestic,168083,0.8447
5,김포,여객,unresolved,4,0.0000
6,김포,화물,domestic,4,1.0000
7,인천,기타,international,923,0.8330
8,인천,기타,domestic,124,0.1119
9,인천,기타,unresolved,61,0.0551


In [9]:
# 원문 대비 고유값 수 변화만 확인한다.
display(query('standardized_value_change'))

,field_name,raw_distinct_count,standardized_distinct_count
0,flight_number,4329,4329
1,airline,128,128
2,delay_reason,20,18


In [10]:
# 원문 지연사유와 표준 사유의 대응을 확인한다.
display(query('delay_reason_standardization_profile'))

,delay_reason_raw,delay_reason_std,is_reason_operational_outcome,rows_count
0,None,None,0,573636
1,항공기 연결에 의한 지연,항공기 연결에 의한 지연,0,99561
2,공항 및 출입국 절차에 의한 지연,공항 및 출입국 절차에 의한 지연,0,42371
3,항공교통흐름에 의한 지연,항공교통흐름에 의한 지연,0,37748
4,여객 및 화물처리에 의한 지연,여객 및 화물처리에 의한 지연,0,11083
5,항공기 정비에 의한 지연,항공기 정비에 의한 지연,0,8845
6,기타 원인에 의한 지연,기타 원인에 의한 지연,0,8812
7,기상에 의한 지연,기상에 의한 지연,0,5705
8,운항 기준 및 승무원 관련 지연,운항 기준 및 승무원 관련 지연,0,3170
9,지상조업 관련 지연,지상조업 관련 지연,0,2226


In [11]:
# 상태값 회항과 지연사유 회항의 관계를 확인한다.
display(query('diversion_definition_crosscheck'))

,airport_name_raw,service_type_raw,status_raw,delay_reason_raw,rows_count
0,김포,여객,취소,회항,145
1,인천,여객,지연,회항,137
2,인천,여객,회항,회항,82
3,김포,여객,회항,회항,80
4,김포,여객,지연,회항,42
5,인천,화물,회항,회항,38
6,인천,화물,지연,회항,20
7,인천,여객,취소,회항,19
8,인천,기타,지연,회항,16
9,김포,화물,지연,회항,3


## 4. 목적지 표준화·국내 공항 참조 검증

In [12]:
# 하나의 표준 목적지에 여러 원문값이 매핑되는 충돌을 확인한다.
destination_conflicts = query('destination_standardization_conflicts')
display(destination_conflicts)

,destination_std,raw_value_count,rows_count,raw_values


In [13]:
# 국제선 실적 유무와 무관한 공식 국내 공항 참조 목록을 확인한다.
display(query('domestic_airport_reference'))

,destination_std,airport_canonical_std,airport_code,reference_source
0,광주,광주,KWJ,KAC official airport list
1,군산,군산,KUV,KAC official airport list
2,김포,김포,GMP,KAC official airport list
3,김해,김해,PUS,KAC official airport list
4,대구,대구,TAE,KAC official airport list
5,무안,무안,MWX,KAC official airport list
6,사천,사천,HIN,KAC official airport list
7,양양,양양,YNY,KAC official airport list
8,여수,여수,RSU,KAC official airport list
9,울산,울산,USN,KAC official airport list


In [14]:
# 검토된 목적지 표기 별칭을 원문과 분리해 확인한다.
display(query('destination_alias_reference'))

,destination_std,destination_canonical_std,scope_std,reference_source
0,가고시마,카고시마,international,reviewed_log_to_route_alias
1,가목사,자무쓰,international,reviewed_log_to_route_alias
2,가오슝,카오슝,international,reviewed_log_to_route_alias
3,고마쓰,고마스,international,reviewed_log_to_route_alias
4,구마모토,구마모도,international,reviewed_log_to_route_alias
...,...,...,...,...
79,하네다,도쿄하네다,international,reviewed_log_to_route_alias
80,하코다테,하꼬다데,international,reviewed_log_to_route_alias
81,항저우,항조우,international,reviewed_log_to_route_alias
82,허페이,허폐,international,reviewed_log_to_route_alias


In [15]:
# 국내선 형태 예외를 제외한 국제선 노선 목적지 참조 목록을 확인한다.
display(query('international_destination_reference'))

,destination_canonical_std,route_destination_raw_values,route_destination_raw_count,reference_source
0,간사이,간사이(KIX),1,v_route_monthly: international and passengers > 0
1,계림,계림(KWL),1,v_route_monthly: international and passengers > 0
2,고마스,고마스(KMQ),1,v_route_monthly: international and passengers > 0
3,고베,고베(UKB),1,v_route_monthly: international and passengers > 0
4,괌,괌(GUM),1,v_route_monthly: international and passengers > 0
...,...,...,...,...
224,화리엔,화리엔(HUN),1,v_route_monthly: international and passengers > 0
225,후에,후에(HUI),1,v_route_monthly: international and passengers > 0
226,후쿠오카,후쿠오카(FUK),1,v_route_monthly: international and passengers > 0
227,후허하오터,후허하오터(HET),1,v_route_monthly: international and passengers > 0


In [16]:
# 모든 목적지의 표준화·별칭·최종 범위 결과를 확인한다.
destination_matching = query('domestic_destination_matching')
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
    display(destination_matching)

,destination_raw,destination_std,destination_canonical_std,flight_scope_std,scope_reference_source,flight_rows
0,신시내티,신시내티,신시내티,unresolved,no_reference_match,4313
1,멤피스,멤피스,멤피스,unresolved,no_reference_match,1911
2,인디애나폴리스,인디애나폴리스,인디애나폴리스,unresolved,no_reference_match,1904
3,마이애미,마이애미,마이애미,unresolved,no_reference_match,1323
4,라이프치히,라이프치히,라이프치히,unresolved,no_reference_match,1180
5,카트비,카트비,카트비,unresolved,no_reference_match,1018
6,루이즈빌,루이즈빌,루이즈빌,unresolved,no_reference_match,806
7,시카고 록퍼드,시카고록퍼드,시카고록퍼드,unresolved,no_reference_match,662
8,페낭,페낭,페낭,unresolved,no_reference_match,540
9,과달라하라,과달라하라,과달라하라,unresolved,no_reference_match,468


In [17]:
# 미확정 목적지는 상위 일부만 자르지 않고 전체를 검증한다.
unresolved_destinations = query('unresolved_destination_profile')
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
    display(unresolved_destinations)

,destination_raw,destination_std,destination_canonical_std,scope_reference_source,flight_rows,passenger_rows
0,신시내티,신시내티,신시내티,no_reference_match,4313,1.0
1,멤피스,멤피스,멤피스,no_reference_match,1911,0.0
2,인디애나폴리스,인디애나폴리스,인디애나폴리스,no_reference_match,1904,0.0
3,마이애미,마이애미,마이애미,no_reference_match,1323,0.0
4,라이프치히,라이프치히,라이프치히,no_reference_match,1180,0.0
5,카트비,카트비,카트비,no_reference_match,1018,1018.0
6,루이즈빌,루이즈빌,루이즈빌,no_reference_match,806,0.0
7,시카고 록퍼드,시카고록퍼드,시카고록퍼드,no_reference_match,662,0.0
8,페낭,페낭,페낭,no_reference_match,540,0.0
9,과달라하라,과달라하라,과달라하라,no_reference_match,468,0.0


In [18]:
# 국내·국제 참조가 동시에 매칭되는 목적지 충돌을 확인한다.
display(query('reference_conflict_profile'))

,destination_raw,destination_std,destination_canonical_std,scope_reference_source,flight_rows,passenger_rows


## 5. 집계 월간 뷰 정합성

In [19]:
# 기간 표준화가 전 행에서 가능했는지 먼저 확인한다.
period_parse_check = query('monthly_period_parse_check')
display(period_parse_check)
if not period_parse_check['period_ym_null_rows'].eq(0).all():
    raise AssertionError('월별 집계 뷰에 period_ym 변환 실패 행이 존재함')

# 기간 표준화 검증은 범위 분류와 무관하게 기존 원문 행을 대상으로 유지한다.

,view_name,rows_count,period_ym_null_rows,first_period_ym,last_period_ym,period_count
0,v_airline_monthly,3843,0.0,202301,202512,36
1,v_airport_monthly,265,0.0,202301,202512,36
2,v_country_monthly,2047,0.0,202301,202512,36
3,v_route_monthly,9634,0.0,202301,202512,36


In [20]:
# 집계 행 제외 후 raw와 월간 뷰 사이의 월별 합계 보존을 확인한다.
monthly_total_check = query('aggregate_view_monthly_total_check')
display(monthly_total_check)
if not monthly_total_check['total_check'].eq('matched').all():
    raise AssertionError('집계 월간 뷰의 raw 대비 월별 합계가 일치하지 않음')

,dataset,period_raw,raw_total,view_total,total_check
0,airline_operations,2023년 01월,13632.0,13632.0,matched
1,airline_operations,2023년 02월,12937.0,12937.0,matched
2,airline_operations,2023년 03월,14407.0,14407.0,matched
3,airline_operations,2023년 04월,15231.0,15231.0,matched
4,airline_operations,2023년 05월,16782.0,16782.0,matched
...,...,...,...,...,...
211,route_passengers,2025년 08월,4131767.0,4131767.0,matched
212,route_passengers,2025년 09월,3794162.0,3794162.0,matched
213,route_passengers,2025년 10월,4043529.0,4043529.0,matched
214,route_passengers,2025년 11월,4025939.0,4025939.0,matched


In [21]:
# 노선 집계의 국내선 형태 예외가 삭제되지 않고 별도 범위로 남는지 확인한다.
display(query('route_scope_profile'))

,route_scope_std,rows_count,passengers,operations
0,international,9632,125773927.0,739998.0
1,domestic,2,0.0,1.0


In [22]:
# 월별 국제선 합계와 국내선 형태 예외·미확정 값을 비교한다.
display(query('route_scope_monthly_comparison'))

,period_ym,all_passengers,international_passengers,domestic_exception_passengers,unresolved_passengers,all_operations,international_operations,domestic_exception_operations,unresolved_operations
0,202301,2403433.0,2403433.0,0.0,0.0,13632.0,13632.0,0.0,0.0
1,202302,2212589.0,2212589.0,0.0,0.0,12937.0,12937.0,0.0,0.0
2,202303,2256529.0,2256529.0,0.0,0.0,14407.0,14407.0,0.0,0.0
3,202304,2412373.0,2412373.0,0.0,0.0,15231.0,15231.0,0.0,0.0
4,202305,2622755.0,2622755.0,0.0,0.0,16782.0,16781.0,1.0,0.0
5,202306,2782311.0,2782311.0,0.0,0.0,16757.0,16757.0,0.0,0.0
6,202307,3269172.0,3269172.0,0.0,0.0,18718.0,18718.0,0.0,0.0
7,202308,3247603.0,3247603.0,0.0,0.0,19138.0,19138.0,0.0,0.0
8,202309,3048940.0,3048940.0,0.0,0.0,18503.0,18503.0,0.0,0.0
9,202310,3171814.0,3171814.0,0.0,0.0,19565.0,19565.0,0.0,0.0


## 6. 전처리 규칙 요약

- `v_flight_analysis`: 원문 12개 컬럼을 유지하고 유효 날짜·월, 유효 시간·분 단위·원시 시간차·기본 플래그·표준화 컬럼을 추가한다.
- `v_departure_flight`: 기존 SQL 호환을 위한 `v_flight_analysis`의 별칭이다. 이후 신규 쿼리는 `v_flight_analysis`를 사용한다.
- `flight_number_std`는 공백 제거·대문자화만, `airline_std`는 앞뒤 공백 제거만 수행한다.
- `destination_std`는 괄호 구간과 공백·슬래시·하이픈·언더스코어를 제거한다. `destination_canonical_std`는 검토된 별칭이 있을 때만 국제선 노선 참조의 표준값을 추가로 보인다. 원문 두 컬럼은 유지한다.
- `v_domestic_airport_reference`는 한국공항공사 14개 공항과 인천국제공항을 포함한 공식 국내 공항 참조다. `v_destination_alias_reference`는 로그와 노선 통계의 검토된 표기 차이를, `v_international_destination_reference`는 국내선 형태 예외를 제외한 양수 국제선 노선 목적지를 보존한다.
- `flight_scope_std`는 `domestic`·`international`·`unresolved`·`reference_conflict`로, `is_international`은 각각 `FALSE`·`TRUE`·`NULL`·`NULL`로 표시한다. 미확정·충돌 행은 삭제하거나 국제선으로 추정하지 않는다.
- 상태·서비스·회항 파생 플래그는 원문 NULL을 `FALSE`로 처리하고, 상태 NULL은 `is_status_unknown`으로 별도 표시한다.
- `delay_reason_std`는 `연결-항공기`만 표준 사유로 치환하고 `회항`은 `NULL`과 운항결과 플래그로 분리한다. `SP`, `98`, `SW`, `ZZ` 등은 원문값을 유지한다.
- `v_route_monthly`는 원문 `period_raw`를 유지하고 조인용 `period_ym`, 노선 범위 `route_scope_std`, `is_international`을 추가한다. 제주–인천 등 국내선 형태 예외 행은 보존하되 국제선 합계에서 별도로 제외할 수 있다.
- 중복 후보는 모두 다른 원문 컬럼을 포함하므로 삭제·제거 플래그를 만들지 않는다.